# NB_01_scenario_demo - Scenario Demo

**Purpose.** Build a CARLA Scenario, run it in synchronous mode, and capture an RGB + ground-truth-risk dataset to disk.

**Inputs.** a CARLA server, a Scenario definition (map, weather, traffic, ego).

**Outputs.** MIREIA/scenarios/<name>/dataset/ (RGB frames + dataset.jsonl per-tick records); optional 2x2 demo video.

**How to run.** Start CARLA (cell 2 launches it), then run cells top-to-bottom. Tweak the Scenario(...) block in section 1 to change the situation.

**Position in the workflow.** Entry point for dataset generation. Precedes NB_04 (training) and NB_05 (feature analysis).


## CARLA Launch

Skip this cell if CARLA is already running.

In [1]:
import subprocess
subprocess.Popen("CarlaUE4.exe", shell=True)

<Popen: returncode: None args: 'CarlaUE4.exe'>

## 0. List available maps
Query the running CARLA server for all available maps.

In [2]:
import carla

client = carla.Client("127.0.0.1", 2000)
client.set_timeout(5.0)

maps = client.get_available_maps()
for m in maps:
    print(m)

print([m.split("/")[-1] for m in maps])

Town01
Town01_Opt
Town02
Town02_Opt
Town03
Town03_Opt
Town04
Town04_Opt
Town05
Town05_Opt
Town06
Town06_Opt
Town07
Town07_Opt
Town10HD
Town10HD_Opt
Town11
Town12
Town13
Town15
AnnotationColorLandscape
['Town01', 'Town01_Opt', 'Town02', 'Town02_Opt', 'Town03', 'Town03_Opt', 'Town04', 'Town04_Opt', 'Town05', 'Town05_Opt', 'Town06', 'Town06_Opt', 'Town07', 'Town07_Opt', 'Town10HD', 'Town10HD_Opt', 'Town11', 'Town12', 'Town13', 'Town15', 'AnnotationColorLandscape']


## 1. Generate scenarios
Generate only missing slots using split-aware logic (base refill towns: 1/2/3/4, with Town05 treated as validation-oriented holdout). Existing scenario folders are preserved.

In [3]:
from MIREIA.config import Config
from MIREIA.simulation.scenarios import generate_mireia_dataset, save_mireia_scenarios
from MIREIA.simulation.world_manager import WorldManager

# Force-fill: any missing base slot is generated as Town04.
scenarios = generate_mireia_dataset(
    target_count=64,
    split_aware=True,
    fill_only_missing_slots=True,
    scenarios_root=Config.PATH_TO_SCENARIOS,
    split_fill_towns=("Town04",),
    split_holdout_towns=("Town05", "Town10HD"),
)
print(f"Generated {len(scenarios)} scenarios to fill missing slots")
for s in scenarios:
    print(f"  - {s.name} ({s.map_name})")

saved, skipped = save_mireia_scenarios(scenarios, overwrite_existing=False)
print(f"Saved: {len(saved)} | Skipped existing: {len(skipped)}")

scenarios_to_run = scenarios[:]  # Run generated scenarios for demonstration
print(f"Prepared scenarios to run: {len(scenarios_to_run)}")

Generated 13 scenarios to fill missing slots
  - 03D_WetNoon_Town04_LowVol (Town04)
  - 06A_MidRainyNoon_Town04_HighVol (Town04)
  - 06B_MidRainyNoon_Town04_LowVol (Town04)
  - 06D_MidRainyNoon_Town04_LowVol (Town04)
  - 09A_CloudySunset_Town04_HighVol (Town04)
  - 09B_CloudySunset_Town04_LowVol (Town04)
  - 09D_CloudySunset_Town04_LowVol (Town04)
  - 12A_SoftRainSunset_Town04_HighVol (Town04)
  - 12B_SoftRainSunset_Town04_LowVol (Town04)
  - 12D_SoftRainSunset_Town04_LowVol (Town04)
  - 15A_ClearNoon_Town04_HighVol_NoFog_Night (Town04)
  - 15B_CloudyNoon_Town04_LowVol_NoFog_Night (Town04)
  - 15D_HardRainNoon_Town04_LowVol_NoFog_Night (Town04)
Saved: 13 | Skipped existing: 0
Prepared scenarios to run: 13


## Capture Configuration

In [ ]:
import time
from pathlib import Path

include_topdown = False
include_risk_maps = False
create_videos = False

frame_stride = 5
# frame_stride=5 at sim dt=0.05s → capture every 0.25s → 4 fps.
# num_frames=3600*5 → 3600 captured frames = 15 min of data.
num_frames = 3600 * frame_stride
frames_per_scenario = num_frames // frame_stride

risk_map_resolution = (1024, 1024)
video_fps = 25
warmup_ticks = 50


def _format_time(seconds: float) -> str:
    return f"{seconds:.1f}s ({seconds / 60:.1f} min)"

## Simulation Loop

Runs each scenario: loads map, warmups, captures frames, saves dataset JSONL, optionally composes video.

> **Stop early**: interrupt the kernel after the scenario you want, or reduce `scenarios_to_run` above.

In [ ]:
from MIREIA.simulation.world_manager import WorldManager

overall_start = time.time()
total_frames_to_capture = frames_per_scenario * len(scenarios_to_run)
total_captured_frames = 0

for idx, scenario in enumerate(scenarios_to_run):
    print(f"\n=== Scenario {idx + 1}/{len(scenarios_to_run)}: {scenario.name} ===")
    scenario_start = time.time()

    # --- Load scenario ---
    wm = WorldManager(verbose=True)
    for attempt in range(5):
        try:
            wm.load_scenario(scenario)
            break
        except Exception as e:
            print(f"Load attempt {attempt + 1} failed: {e}")
            if attempt == 4:
                raise

    for _ in range(warmup_ticks):
        wm.tick()

    # --- Prepare output paths ---
    sensors = wm.setup_sensors()
    scenario_root = Path(wm.scenario.folder_path)
    images_dir = scenario_root / "dataset" / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    wm.enable_recording(append=False, include_topdown=include_topdown, include_static_risk_image=False)

    def _tick_silent():
        wm.world.tick()
        if wm.bridge is not None:
            wm.bridge.update()

    # --- Frame capture loop ---
    captured_frames = 0
    for frame_id in range(num_frames):
        if frame_id % frame_stride != 0:
            continue

        rgb_path = images_dir / f"rgb_{frame_id:06d}.png"
        rel_rgb = str(rgb_path.relative_to(scenario_root))
        topdown_rel = risk_map_rel = ""

        if include_topdown:
            topdown_path = images_dir / f"topdown_{frame_id:06d}.png"
            topdown_rel = str(topdown_path.relative_to(scenario_root))
        if include_risk_maps:
            risk_map_path = images_dir / f"risk_{frame_id:06d}.png"
            risk_map_rel = str(risk_map_path.relative_to(scenario_root))

        def _tick_and_log():
            wm.tick(ground_truth_risk=None, rgb_image_path=rel_rgb,
                    topdown_image_path=topdown_rel, risk_map_image_path=risk_map_rel)

        sensors.save_ego_frame(save_path=str(rgb_path), tick_fn=_tick_and_log)

        if include_topdown:
            sensors.save_map_frame(save_path=str(topdown_path), tick_fn=_tick_silent)
        if include_risk_maps:
            wm.save_risk_frame_image(save_path=str(risk_map_path), resolution=risk_map_resolution)

        captured_frames += 1
        total_captured_frames += 1

        if captured_frames % 50 == 0:
            scen_elapsed = time.time() - scenario_start
            avg_fps = captured_frames / max(scen_elapsed, 1e-9)
            print(
                f"  [{frame_id}/{num_frames}] "
                f"scenario remaining: {_format_time((frames_per_scenario - captured_frames) / max(avg_fps, 1e-9))} | "
                f"overall: {total_captured_frames}/{total_frames_to_capture}"
            )

    # --- Teardown ---
    print(f"  Dataset saved to: {scenario_root / 'dataset'}")
    if create_videos:
        video_path = wm.compose_dataset_video(fps=video_fps)
        print(f"  Video saved to: {video_path}")

    wm.destroy()

print(f"\nAll scenarios done in {_format_time(time.time() - overall_start)}.")

## Post-Processing

The cells below copy videos to a central folder and compose a 2×2 grid video.
Run independently of the simulation loop above.

In [ ]:
from pathlib import Path
import shutil

scenarios_root = Path("MIREIA/scenarios")
videos_out = scenarios_root / "videos"
videos_out.mkdir(parents=True, exist_ok=True)

video_name = "dataset_video.mp4"
copied = 0
for scenario_dir in scenarios_root.iterdir():
    if not scenario_dir.is_dir() or scenario_dir.name == "videos":
        continue
    src = scenario_dir / "dataset" / video_name
    if src.exists():
        dst = videos_out / f"{scenario_dir.name}_{video_name}"
        shutil.copy2(src, dst)
        copied += 1

print(f"Copied {copied} videos to: {videos_out}")

Copied 54 videos to: D:\Projectes\TFG\MIREIA\scenarios\videos


: 

: 

: 

### 2×2 Grid Composition (FFmpeg)

In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import tempfile

videos_dir = Path("MIREIA/scenarios/videos")
output_path = videos_dir / "all_scenarios_grid_concat.mp4"

videos = sorted(videos_dir.glob("*_dataset_video.mp4"))
if not videos:
    raise RuntimeError(f"No videos found in {videos_dir}")

groups = {}
for video in videos:
    scenario_name = video.name.replace("_dataset_video.mp4", "")
    match = re.match(r"(\d{2})([A-D])_", scenario_name)
    if not match:
        continue
    group_key, letter = match.group(1), match.group(2)
    groups.setdefault(group_key, {})[letter] = video

temp_dir = videos_dir / "_grid_videos"
if temp_dir.exists():
    shutil.rmtree(temp_dir)
temp_dir.mkdir(parents=True, exist_ok=True)

grid_videos = []
for group_key in sorted(groups.keys()):
    slots = groups[group_key]
    if not all(k in slots for k in ("A", "B", "C", "D")):
        print(f"Skipping {group_key}: missing A/B/C/D")
        continue
    out_path = temp_dir / f"grid_{group_key}.mp4"
    subprocess.run([
        "ffmpeg", "-y",
        "-i", str(slots["A"]), "-i", str(slots["B"]),
        "-i", str(slots["C"]), "-i", str(slots["D"]),
        "-filter_complex", "[0:v][1:v][2:v][3:v]xstack=inputs=4:layout=0_0|w0_0|0_h0|w0_h0[v]",
        "-map", "[v]", "-shortest",
        "-c:v", "libx264", "-preset", "fast", "-crf", "20", "-an",
        str(out_path),
    ], check=True)
    grid_videos.append(out_path)

if not grid_videos:
    raise RuntimeError("No complete A/B/C/D groups found for grid composition.")

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    for video in grid_videos:
        f.write(f"file '{video.as_posix()}'\n")
    concat_list = f.name

subprocess.run([
    "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", concat_list,
    "-c:v", "libx264", "-preset", "fast", "-crf", "20", "-movflags", "+faststart",
    str(output_path),
], check=True)

print(f"Saved grid video to: {output_path}")